# Week 9 · Day 2 — The Same MNIST Classifier, Now in PyTorch

**Yesterday** you built a handwritten-digit classifier in **TensorFlow/Keras** — loaded MNIST, flattened it, built a `784 → 128 → 64 → 10` network, trained it, and got strong accuracy.

**Today** we build *the exact same classifier in PyTorch* — same data, same architecture, same result. The only thing that changes is the framework. PyTorch is our main tool for the rest of the course, so today is about getting comfortable in it, slowly and carefully.

And while we're here, we'll dig into **two ideas that decide whether any network trains well:**
1. **Activation functions** — the little non-linear switch inside every neuron. We'll see *why* they matter and how ReLU, sigmoid, tanh differ.
2. **Optimizers** — the rule that updates the weights. We'll compare SGD, Momentum, and Adam and *watch* the difference.

**Today's plan:**
1. Load MNIST from disk with `os` (the real-project way).
2. Prepare the data (same steps as yesterday).
3. **Learn PyTorch gently** — tensors, a model, the training loop, piece by piece.
4. Train the classifier and evaluate it.
5. **Deep dive: activation functions.**
6. **Deep dive: optimizers.**

> You already know the *concepts* from yesterday. Today you’re learning the *PyTorch words* for them. Go slow, run every cell.

---
## 1. Load the data with `os`

Real projects don’t hand you a clean `load_data()` — the data sits in **files in folders**, and you build the paths yourself. We’ll use Python’s **`os`** module to join folder + filename into a full path. This is a habit worth building.

MNIST comes as four **idx** files (the same ones from yesterday): train images, train labels, test images, test labels. We read them with `idx2numpy`.

In [ ]:
# install if needed
# !pip install idx2numpy

import os
import numpy as np
import idx2numpy
import matplotlib.pyplot as plt

# ---- point this to the folder that holds the 4 idx files ----
# os.path.join builds a correct path on any OS (Windows \\ vs Linux /)
DATA_DIR = os.path.join("data", "mnist")   # e.g. ./data/mnist/

# build the four full file paths with os.path.join
train_images_path = os.path.join(DATA_DIR, "train-images.idx3-ubyte")
train_labels_path = os.path.join(DATA_DIR, "train-labels.idx1-ubyte")
test_images_path  = os.path.join(DATA_DIR, "t10k-images.idx3-ubyte")
test_labels_path  = os.path.join(DATA_DIR, "t10k-labels.idx1-ubyte")

# a quick check that the files are actually there before we try to read them
for p in [train_images_path, train_labels_path, test_images_path, test_labels_path]:
    print("found" if os.path.exists(p) else "MISSING", "->", p)

> If any file says **MISSING**: check that `DATA_DIR` points to the folder containing the four `idx` files. On Kaggle it might be something like `/kaggle/input/mnist-dataset/`. The whole point of `os.path.join` is that you only fix the folder once, up top.

In [ ]:
# read the idx files into numpy arrays
X = idx2numpy.convert_from_file(train_images_path)
y = idx2numpy.convert_from_file(train_labels_path)
X_test = idx2numpy.convert_from_file(test_images_path)
y_test = idx2numpy.convert_from_file(test_labels_path)

print("train images:", X.shape)      # (60000, 28, 28)
print("train labels:", y.shape)      # (60000,)
print("test images: ", X_test.shape) # (10000, 28, 28)
print("test labels: ", y_test.shape)

In [ ]:
# look at the data first (always)
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X[i], cmap="gray")
    plt.title(f"label: {y[i]}")
    plt.axis("off")
plt.suptitle("MNIST — same digits as yesterday")
plt.tight_layout()
plt.show()

---
## 2. Prepare the data (same steps as yesterday)

Exactly the workflow from the Keras notebook — nothing framework-specific yet:
1. **Flatten** each 28×28 image to a 784-vector.
2. **Split** train into train + validation.
3. **Scale** with `StandardScaler` (fit on train only).

We’ll convert to PyTorch tensors *after* this, as the final step.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. flatten 28x28 -> 784
X = X.reshape(X.shape[0], -1).astype("float32")
X_test = X_test.reshape(X_test.shape[0], -1).astype("float32")

# 2. split off a validation set
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y)

# 3. scale (fit on train only, apply to all)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

---
## 3. Learning PyTorch, gently

Here’s where PyTorch begins. We’ll go one concept at a time. Nothing here is new *thinking* — it’s the same network as yesterday — just new vocabulary.

### 3a. Tensors
A **tensor** is PyTorch’s version of a NumPy array. Same idea (a grid of numbers), with two extras: it can live on a GPU, and it can track gradients for training. Converting NumPy → tensor is one line.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# convert our prepared numpy arrays into tensors
# images -> float tensors ; labels -> long (integer) tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

print("a tensor looks just like an array:")
print("  shape:", X_train_t.shape)
print("  dtype:", X_train_t.dtype)
print("  first label:", y_train_t[0].item())

> **Why labels are `long`, not one-hot.** Yesterday in Keras you one-hot encoded the labels (`to_categorical`) and used `categorical_crossentropy`. PyTorch is simpler here: keep the labels as plain integers (`0`–9) and use `CrossEntropyLoss`, which does the one-hot step internally. One less thing to do.

### 3b. Batches with `DataLoader`
Training on all 54,000 images at once is wasteful. We feed the network **small batches** (128 images at a time), just like `batch_size=128` in yesterday’s `model.fit`. PyTorch’s `DataLoader` is the batch server that hands them out, shuffled.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

print("batches per epoch:", len(train_loader), "(each has up to 128 images)")

### 3c. The model
Yesterday’s Keras model was:
```python
Sequential([Input(784), Dense(128, 'relu'), Dense(64, 'relu'), Dense(10, 'softmax')])
```
PyTorch’s `nn.Sequential` is almost the same, with three small differences to notice:
- `nn.Linear(784, 128)` is Keras’s `Dense(128)` — you state **both** the in-size and out-size.
- ReLU is its own layer, `nn.ReLU()`, placed *after* each Linear.
- **No softmax at the end.** `CrossEntropyLoss` applies it internally, so the last layer outputs raw scores.

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 128),   # Dense(128) — 784 in, 128 out
    nn.ReLU(),
    nn.Linear(128, 64),    # Dense(64)
    nn.ReLU(),
    nn.Linear(64, 10)      # Dense(10) — raw scores, no softmax
)

print(model)

### 3d. Loss and optimizer
Two choices, exactly like Keras’s `compile(loss=..., optimizer=...)` — just as separate objects:
- **loss:** `nn.CrossEntropyLoss()` (multi-class; expects integer labels + raw scores).
- **optimizer:** `Adam`, same as yesterday. It’s the rule that updates the weights. (We’ll dissect optimizers in Part 6.)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("loss and optimizer ready")

### 3e. The training loop — the one real difference from Keras
In Keras, `model.fit()` hid the loop. In PyTorch **you write it** — which is exactly why we use PyTorch to learn: the four moves stay visible.

The **four moves**, once per batch:
1. **forward** — `preds = model(xb)`
2. **loss** — `loss = loss_fn(preds, yb)`
3. **backward** — `optimizer.zero_grad()` then `loss.backward()`
4. **update** — `optimizer.step()`

We wrap this in a reusable function so we can reuse it later when we compare optimizers.

In [ ]:
def train_model(model, optimizer, loss_fn, loader,
                X_val_t, y_val_t, epochs=15, log=True):
    """Train a model and return per-epoch train loss + val accuracy."""
    train_losses, val_accs = [], []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in loader:
            preds = model(xb)                 # 1. forward
            loss = loss_fn(preds, yb)         # 2. loss
            optimizer.zero_grad()             # 3. backward...
            loss.backward()
            optimizer.step()                  # 4. update
            running += loss.item()
        train_losses.append(running / len(loader))

        # check validation accuracy each epoch
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t).argmax(dim=1)
            val_acc = (val_pred == y_val_t).float().mean().item()
        val_accs.append(val_acc)
        if log:
            print(f"epoch {epoch+1:2d}  train loss {train_losses[-1]:.4f}  val acc {val_acc:.2%}")
    return train_losses, val_accs

---
## 4. Train and evaluate the classifier

Now run it. This trains the same network you built yesterday — in PyTorch.

In [ ]:
train_losses, val_accs = train_model(
    model, optimizer, loss_fn, train_loader, X_val_t, y_val_t, epochs=15)

In [ ]:
# the two curves, like yesterday's accuracy/loss graphs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, color="purple", marker="o")
ax1.set_title("Training loss"); ax1.set_xlabel("epoch"); ax1.grid(alpha=0.3)
ax2.plot(val_accs, color="green", marker="o")
ax2.set_title("Validation accuracy"); ax2.set_xlabel("epoch"); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# final accuracy on the held-out test set
model.eval()
with torch.no_grad():
    test_pred = model(X_test_t).argmax(dim=1)
test_acc = (test_pred == y_test_t).float().mean().item()
print(f"TEST ACCURACY: {test_acc:.2%}")

In [ ]:
# confusion matrix — same evaluation habit as always
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_t, test_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Confusion matrix — {test_acc:.1%}")
plt.show()

In [ ]:
# look at some predictions — green = correct, red = wrong
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    # undo scaling just for display
    img = scaler.inverse_transform(X_test[i:i+1]).reshape(28, 28)
    plt.imshow(img, cmap="gray")
    t, p = y_test_t[i].item(), test_pred[i].item()
    plt.title(f"pred {p} / true {t}", color="green" if t == p else "red", fontsize=10)
    plt.axis("off")
plt.suptitle("PyTorch predictions")
plt.tight_layout()
plt.show()

**Same result as yesterday, in PyTorch.** You loaded data with `os`, prepared it, built a network, wrote the training loop by hand, and evaluated it. That’s a complete PyTorch project.

Now the two deep-dive concepts — the things that decide whether a network like this trains *well*.

---
## 5. Deep dive: Activation functions

An **activation function** is the small non-linear step inside each neuron, applied to its weighted sum. Every `nn.ReLU()` in our model is one.

### Why do we even need them?
Here’s the key idea: **without** an activation, a neural network — no matter how many layers — is just a stack of straight-line (linear) operations, which collapses into *one* straight line. It could never learn a curve, or separate digits. The activation adds the **bend** that lets the network model complex patterns. (Remember XOR in Week 8 — it needed that bend.)

Let’s look at the three you’ll meet most.

In [ ]:
z = torch.linspace(-6, 6, 200)   # a range of inputs

relu = torch.relu(z)
sigmoid = torch.sigmoid(z)
tanh = torch.tanh(z)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, out, desc) in zip(axes, [
    ("ReLU",    relu,    "0 for negatives, straight line for positives"),
    ("Sigmoid", sigmoid, "squashes everything into (0, 1)"),
    ("Tanh",    tanh,    "squashes everything into (-1, 1)")]):
    ax.plot(z, out, color="purple", linewidth=2)
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_title(name); ax.set_xlabel(desc, fontsize=9); ax.grid(alpha=0.3)
plt.suptitle("The three activation functions you'll use most")
plt.tight_layout()
plt.show()

### How they differ, and when to use them

| Activation | Output range | Good for | Watch out for |
|---|---|---|---|
| **ReLU** | 0 to ∞ | **hidden layers** (the modern default) | “dying” neurons stuck at 0 |
| **Sigmoid** | 0 to 1 | a **binary** output (probability) | saturates — slows learning in deep nets |
| **Tanh** | −1 to 1 | hidden layers (older nets, RNNs) | also saturates at the extremes |

**The practical rule:** use **ReLU** in hidden layers unless you have a reason not to. Use **sigmoid** on the *output* for binary yes/no. For multi-class output (like our digits) we use no activation on the last layer and let `CrossEntropyLoss` handle it.

### See it matter: swap ReLU for sigmoid in the hidden layers
Let’s train the *same network* but with **sigmoid** hidden activations and watch it learn more slowly — a real demonstration of why ReLU became the default.

In [ ]:
def make_model(activation):
    """Same 784->128->64->10 net, but with a chosen hidden activation."""
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(784, 128), activation(),
        nn.Linear(128, 64),  activation(),
        nn.Linear(64, 10))

results_act = {}
for name, act in [("ReLU", nn.ReLU), ("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh)]:
    m = make_model(act)
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    _, accs = train_model(m, opt, loss_fn, train_loader, X_val_t, y_val_t, epochs=10, log=False)
    results_act[name] = accs
    print(f"{name:8s} final val acc: {accs[-1]:.2%}")

In [ ]:
plt.figure(figsize=(9, 5))
for name, accs in results_act.items():
    plt.plot(range(1, len(accs)+1), accs, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("validation accuracy")
plt.title("Activation functions compared (same network, same data)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Read the curves: **ReLU and tanh usually climb fast**, while **sigmoid often lags early** — its gradients are small when inputs are large (saturation), so weights update slowly. On a short 10-epoch run that gap is exactly why modern networks default to ReLU in hidden layers. *(Numbers vary a little per run — the pattern is the lesson, not the exact values.)*

---
## 6. Deep dive: Optimizers

The **optimizer** is the rule for updating weights after `loss.backward()` computes the gradients. Yesterday you just wrote `optimizer="adam"`. Today we open the box.

All optimizers do the same job — *nudge each weight to reduce the loss* — but they differ in **how** they take that step:

- **SGD** (plain): step directly downhill by the gradient × learning rate. Simple, but can be slow and zig-zaggy.
- **SGD + Momentum:** remember the previous direction and keep some speed — like a ball rolling downhill. Smooths out the zig-zag, moves faster.
- **Adam:** adapts the step size for *each* weight automatically, using both momentum and a running estimate of gradient size. Usually the fastest to get going and the safest default.

Let’s train the same network three times — one per optimizer — and watch them race.

In [ ]:
def make_relu_model():
    torch.manual_seed(42)
    return nn.Sequential(
        nn.Linear(784, 128), nn.ReLU(),
        nn.Linear(128, 64),  nn.ReLU(),
        nn.Linear(64, 10))

# same model, same data, three different optimizers
optimizers = {
    "SGD":          lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD+Momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam":         lambda p: torch.optim.Adam(p, lr=0.001),
}

results_opt = {}
for name, make_opt in optimizers.items():
    m = make_relu_model()
    opt = make_opt(m.parameters())
    losses, _ = train_model(m, opt, loss_fn, train_loader, X_val_t, y_val_t, epochs=10, log=False)
    results_opt[name] = losses
    print(f"{name:14s} final train loss: {losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(9, 5))
for name, losses in results_opt.items():
    plt.plot(range(1, len(losses)+1), losses, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("training loss")
plt.title("Optimizers compared (same network, same data)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

Read the curves: **plain SGD** usually falls the slowest, **Momentum** speeds it up, and **Adam** typically drops fastest in these early epochs. That’s why Adam is the common default — it “just works” without much tuning. But note: on some problems a well-tuned SGD+Momentum generalizes *better*, so Adam isn’t always the final answer. Knowing the trade-off is the skill.

### The one knob they all share: learning rate
Every optimizer has a **learning rate** — the step size. Too big and it overshoots (loss explodes or wobbles); too small and it crawls. It’s the single most important number to get roughly right, whatever optimizer you pick.

In [ ]:
# Adam at three learning rates — see the effect
for lr in [0.0001, 0.001, 0.01, 0.1]:
    m = make_relu_model()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    losses, accs = train_model(m, opt, loss_fn, train_loader, X_val_t, y_val_t, epochs=5, log=False)
    print(f"lr={lr:<7}  final train loss {losses[-1]:.4f}   val acc {accs[-1]:.2%}")

Typically the middle values (0.001–0.01) do best; 0.0001 underfits in only 5 epochs, and 0.1 is often too aggressive and unstable. This is the same learning-rate lesson from Week 8 — now across optimizers.

---
## Your turn (practice) ✍️

You’ve seen it all demonstrated — now make small changes and observe. Pick at least two:

1. **Add a third hidden layer** to the model (e.g. `784 → 256 → 128 → 64 → 10`). Does test accuracy improve?
2. **Try `nn.LeakyReLU()`** as the activation (it fixes ReLU’s “dying neuron” problem). Compare it to ReLU on the 10-epoch chart.
3. **Train longer** with Momentum (20–30 epochs) — does it catch up to Adam?
4. **Push the learning rate** to 0.5 with Adam and watch what breaks.

Write one or two sentences under each about what you saw.

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====
# Reuse make_relu_model() / make_model() and train_model() from above.
# Change ONE thing at a time and compare.



---
## Summary

**PyTorch, learned by rebuilding yesterday’s classifier:**
- **`os.path.join`** builds file paths the real-project way — fix the folder once, up top.
- A **tensor** is a NumPy array that can train; labels stay **integer** (`long`) with `CrossEntropyLoss` (no one-hot needed).
- `nn.Sequential` + `nn.Linear` + `nn.ReLU` is Keras’s `Sequential`/`Dense` — but **no softmax** on the last layer.
- The **training loop is yours to write**: forward → loss → backward → update, over batches from a `DataLoader`.

**Activation functions** — the non-linear bend inside each neuron:
- Without them, a deep network collapses to a single straight line.
- **ReLU** for hidden layers (the default), **sigmoid** for binary output, **tanh** as an older alternative. We *saw* ReLU learn faster than sigmoid.

**Optimizers** — the weight-update rule:
- **SGD** (slow, simple) → **+Momentum** (faster, smoother) → **Adam** (adaptive, best default). We *watched* Adam’s loss fall fastest.
- Every optimizer shares one critical knob: the **learning rate**.

You now have a real, working PyTorch project and a feel for the two settings that make or break training. **Next**, we take PyTorch forward into the rest of Week 9.